In [1]:
from ssb import *
from utils.universal_data_loader import *

# Example Feature Groupings Setup
# business_groups = {
# "delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m", "max_dpd_12m"],            
# "transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
# "spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
# "repayment_vars":["payment_ratio_avg_3m", "payment_ratio_avg_6m", "payment_ratio_avg_12m"],
# "card_utilization_vars":["utilization_avg_3m", "utilization_avg_6m", "utilization_avg_12m", "utilization_max_12m"],
# }

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000, 5000, 1000, 500],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="Default_Status",
    n_jobs=-1,
    min_sample_size=5000,
    min_lift=1.5,
    top_n_vars=5,
    max_segments=10,
    enable_diversity=False,
    max_feature_reuse=2,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=None, #business_groups,
    ignore_features=None,
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/Strategic_Segment_Builder/Datasets/credit_card_default_2.5M.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-08 16:00:52,290 | INFO     | [universal_data_loader.py:74] | Detecting local file format handler for extension: '.parquet'...
2026-07-08 16:00:52,433 | INFO     | [ssb.py:364] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-08 16:00:52,477 | INFO     | [ssb.py:381] | Dynamic Grid Search Enabled: 8 total configurations.
2026-07-08 16:00:52,480 | INFO     | [ssb.py:402] | Iteration 1 | Remaining Volume: 250,000 | Base Rate: 26.02%
2026-07-08 16:00:59,810 | INFO     | [ssb.py:502] | Feature Usage Tracker Update -> 'Employment_Type' used count = 1
2026-07-08 16:00:59,811 | INFO     | [ssb.py:502] | Feature Usage Tracker Update -> 'Missed_Payment_Pattern' used count = 1
2026-07-08 16:00:59,812 | INFO     | [ssb.py:502] | Feature Usage Tracker Update -> 'Housing_Status' used count = 1
2026-07-08 16:00:59,812 | INFO     | [ssb.py:517] | Segment 1 Captured (Size Floor: 1000 | Lift Floor: 1.5): Employment_Type IN ('Unemployed') AND Missed_Payment_Pattern IN ('Frequent') AND Ho

In [2]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate    |        lift        |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+
|   0.0   |   216930.0  |    50589.0    | 23.320425943852857 | 26.024399999999996 |       86.772       | 0.8960985053969682 |
|   1.0   |    1303.0   |     1295.0    | 99.38603223330774  | 26.024399999999996 |       0.5212       | 3.818955758184925  |
|   2.0   |    2056.0   |     1993.0    | 96.93579766536965  | 26.024399999999996 |       0.8224       | 3.7248043246095843 |
|   3.0   |   29711.0   |    11184.0    | 37.642623943993804 | 26.024399999999996 | 11.884400000000001 | 1.446435804245009  |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------

In [3]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Employment_Type=<ArrowStringArray>
['Unemployed']
Length: 1, dtype: str & Missed_Payment_Pattern=<ArrowStringArray>
['Frequent']
Length: 1, dtype: str & Housing_Status=<ArrowStringArray>
['Other', 'Rent']
Length: 2, dtype: str
SQL Filter: Employment_Type IN ('Unemployed') AND Missed_Payment_Pattern IN ('Frequent') AND Housing_Status IN ('Other', 'Rent')
--------------------------------------------------
Segment ID: 2
Raw Rule:   Employment_Type=<ArrowStringArray>
['Self-Employed']
Length: 1, dtype: str & Housing_Status=<ArrowStringArray>
['Other', 'Rent']
Length: 2, dtype: str & Missed_Payment_Pattern=<ArrowStringArray>
['Frequent']
Length: 1, dtype: str
SQL Filter: Employment_Type IN ('Self-Employed') AND Housing_Status IN ('Other', 'Rent') AND Missed_Payment_Pattern IN ('Frequent')
--------------------------------------------------
Segment ID: 3
Raw Rule:   Prior_Bankruptcy=<ArrowStringArray>
['Yes']
Length: 1, dtype: str
SQL Filt

In [4]:
pd.DataFrame(final_eval)

,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,216930,50589.0,23.320426,26.0244,86.7720,0.896099
1,1,1303,1295.0,99.386032,26.0244,0.5212,3.818956
2,2,2056,1993.0,96.935798,26.0244,0.8224,3.724804
3,3,29711,11184.0,37.642624,26.0244,11.8844,1.446436


In [5]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
            SELECT *, 
            CASE WHEN (Study_Hours >= 4.14 AND Study_Hours < 5.38) AND Exam_Pressure >= 8.50
            THEN 1 ELSE 0 END AS seg_1,
            CASE WHEN Exam_Pressure >= 8.50
            THEN 1 ELSE 0 END AS seg_2,
            CASE WHEN Study_Hours >= 8.44
            THEN 1 ELSE 0 END AS seg_3,
            CASE WHEN (Sleep_Hours >= 2.04 AND Sleep_Hours < 3.96)
            THEN 1 ELSE 0 END AS seg_4,
            FROM predicted
""").df()
conn.close()

In [6]:
scorer = StrategicSegmentScore(
    target_col="Stress_Level",
    primary_key="index",
    segment_cols=["seg_1", "seg_2", "seg_3", "seg_4"],
)

In [7]:
scorer.calculate_and_export_weights(predicted, export_path="scorecard_model_test1.json")

2026-07-06 12:40:00,116 | INFO     | [ssb.py:556] | Initializing DuckDB scorecard engine...
2026-07-06 12:40:00,163 | INFO     | [ssb.py:591] | Computing harmonic scorecard weights...
2026-07-06 12:40:00,164 | INFO     | [ssb.py:628] | Scoring training dataset via NumPy Linear Algebra engine...
2026-07-06 12:40:00,167 | INFO     | [ssb.py:645] | Dataset Zero-Inflation Rate: 70.01%
2026-07-06 12:40:00,168 | INFO     | [ssb.py:651] | Normal Distribution (<80%). Deciling across full dataset...
2026-07-06 12:40:00,168 | INFO     | [ssb.py:660] | Calibrating deciles across 25,500 target customers...


{'model_metadata': {'total_training_population': 25500,
  'active_scored_population': 25500,
  'active_population_pct': 100.0,
  'baseline_event_rate': 0.2999},
 'segment_weights': {'seg_1': {'weight': 31,
   'lift': 2.828,
   'response_rate': 0.8481,
   'capture_rate': 0.0577},
  'seg_2': {'weight': 117,
   'lift': 2.752,
   'response_rate': 0.8253,
   'capture_rate': 0.2853},
  'seg_3': {'weight': 33,
   'lift': 1.9159,
   'response_rate': 0.5745,
   'capture_rate': 0.1003},
  'seg_4': {'weight': 21,
   'lift': 1.5182,
   'response_rate': 0.4553,
   'capture_rate': 0.0812}},
 'decile_min_thresholds': {'1': 117,
  '2': 0,
  '3': 0,
  '4': 0,
  '5': 0,
  '6': 0,
  '7': 0,
  '8': 0,
  '9': 0,
  '10': 0}}

In [8]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/Scores/scorecard_model_test1.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-06 12:40:00,187 | INFO     | [1327506462.py:2] | Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/Scores/scorecard_model_test1.json...


In [9]:
model_artifact.get("decile_min_thresholds")

{'1': 117,
 '2': 0,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [10]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 31
Segment: seg_2 | Weight: 117
Segment: seg_3 | Weight: 33
Segment: seg_4 | Weight: 21


In [11]:
model_artifact.get("decile_min_thresholds")

{'1': 117,
 '2': 0,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [12]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 31 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 117 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 33 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 21 ELSE 0 END AS seg_4_weighted
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=117 THEN 1
                    WHEN total_weight >= 0 THEN 2
                    WHEN total_weight >= 0 THEN 3
                    WHEN total_weight >= 0 THEN 4
                    WHEN total_weight >= 0 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [13]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(Stress_Level) AS events, 
                    (SUM(Stress_Level)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()

In [14]:
scored

,decile_band,count,events,response_rate
0,1,2644,2182.0,82.526475
1,2,22856,5465.0,23.910571
